In [34]:
# импортируем библиотеки
import pandas as pd
from datetime import datetime, date

# словари макрорегионов и федеральных округов + утилиты нормализации
from macroregions import COUNTRY_MACRO_REGIONS
from russian_fo import RUSSIAN_REGIONS

In [35]:
events_name = f"events {datetime.now().strftime('%Y-%m-%d %H-%M')}"
essay_name = f"essay {datetime.now().strftime('%Y-%m-%d %H-%M')}"
video_name = f"video {datetime.now().strftime('%Y-%m-%d %H-%M')}"

print(events_name)
print(essay_name)
print(video_name)

events 2026-03-16 16-41
essay 2026-03-16 16-41
video 2026-03-16 16-41


In [36]:
try:
    df = pd.read_csv(f'../data/users.csv', sep=';')
except:
    df = pd.read_csv(f'../data/users.csv', sep=';')

C:\Users\glebn\AppData\Local\Temp\ipykernel_10852\3934597423.py:2: DtypeWarning: Columns (25,39,129,131,168,185,187,198,202,219,222,232,234,238,239,240,241,242,255,271,274,277) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'../data/users.csv', sep=';')


In [37]:
col = [
    'ID',
    'Email #email',
    'Гражданство',
    'Страна проживания',
    'Регион проживания',
    'Возраст',
    'Состояние',
    'Готов добраться за свой счёт #travelownexpense',
    'Участие в РП (да/нет) #expeditionprogram',
    'Дата создания заявки',
    'Имя на русском #firstName.ru',
    'Имя на английском #firstName.en',
    'Нет имени',
    'Фамилия на русском #lastName.ru',
    'Фамилия на английском #lastName.en',
    'Нет фамилии',
    'Отчество на русском #patronymic.ru',
    'Отчество на английском #patronymic.en',
    'Нет отчества',
    'Пол #sex',
    'Дата рождения',
    'Адрес фактического места проживания #addressresidence',
    'Родной язык #nativelanguage',
    'Род деятельность #statusactivity',
    'Направление деятельности #fieldwork',
    'Номер телефона #phone',
    'Фотография #photo',
    'Город фактического места проживания #cityresidence',
    'Язык переписки #correspondencelanguage',
    'Тип питания #dietype',
    'Уровень владения английским языком #englevel',
    'Уровень владения русским языком #ruslevel'
    ]

In [38]:
df = df[col].copy()

In [39]:
try:
    df_video = pd.read_csv(f'../data/video.csv', sep=';')
    df_essay = pd.read_csv(f'../data/esse.csv', sep=';')
except:
    df_video = pd.read_csv(f'../data/video.csv', sep=';')
    df_essay = pd.read_csv(f'../data/esse.csv', sep=';')

In [40]:
# Фильтрация email с @wyffest.com
df = df[~df['Email #email'].str.contains('@wyffest.com', case=False, na=False)]
# Убираем отозванные заявки
df = df[(df['Состояние'] == "Черновик") | (df['Состояние'] == "На рассмотрении")]

In [41]:
# Подготовка данных для основной статистики
df['is_child'] = df['Возраст'].between(14, 17)
df['is_rf_citizenship'] = df['Гражданство'] == 'Россия'
df['is_rf_residence'] = df['Страна проживания'] == 'Россия'

# Фильтрация эссе и видео по статусу "Отправлена"
df_essay_filtered = df_essay[df_essay['Состояние'] == 'Отправлена'].copy()
df_video_filtered = df_video[df_video['Состояние'] == 'Отправлена'].copy()

# Убираем @wyffest.com из эссе и видео
df_essay_filtered = df_essay_filtered[~df_essay_filtered['Автор'].str.contains('@wyffest.com', case=False, na=False)]
df_video_filtered = df_video_filtered[~df_video_filtered['Автор'].str.contains('@wyffest.com', case=False, na=False)]

# Оставляем только уникальных авторов
df_essay_unique = df_essay_filtered.drop_duplicates(subset=['Автор'])
df_video_unique = df_video_filtered.drop_duplicates(subset=['Автор'])

In [42]:
# # Подтягиваем возраст и категории из основной df
# df_essay_merged = df.merge(
#     df_essay_unique[['Автор', 'Состояние']],
#     left_on='Email #email',
#     right_on='Автор',
#     how='left'
# )

# df_essay_merged = df_essay_merged.rename(columns={
#     'Состояние_y':'Статус эссе'
# })

# df_essay_merged = df_essay_merged.drop(columns={"Автор"})

In [43]:
# Присоединяем к уникальной заявке статус эссе и статус видеовизитки.
# Если эссе или видео нет, то присоединится NaN.
df_merged = df.drop_duplicates(subset=['Email #email']).merge(
    df_essay_unique[['Автор', 'Состояние']].rename(columns={'Автор': 'Email #email', 'Состояние': 'Статус эссе'}),
    on='Email #email',
    how='left'
).merge(
    df_video_unique[['Автор', 'Состояние']].rename(columns={'Автор': 'Email #email', 'Состояние': 'Статус видеовизитки'}),
    on='Email #email',
    how='left'
)

# Статус заявки нам нужен из основной df (там это "Состояние"), поэтому добавим явно:
df_merged['Статус заявки'] = df_merged['Состояние']
df_merged = df_merged.drop(columns=['Состояние'])

In [44]:
# Общая статистика
total = len(df_merged)
children_14_17 = df_merged['is_child'].sum()
adults_18_35 = total - children_14_17

# Общая статистика по эссе и видео
total_essay = len(df_merged[df_merged['Статус эссе'].notna()])
essay_children = df_merged[df_merged['Статус эссе'].notna() & df_merged['is_child']].shape[0]
essay_adults = total_essay - essay_children

total_video = len(df_merged[df_merged['Статус видеовизитки'].notna()])
video_children = df_merged[df_merged['Статус видеовизитки'].notna() & df_merged['is_child']].shape[0]
video_adults = total_video - video_children

In [45]:
# РФ
rf = df_merged[(df_merged['is_rf_citizenship']) & (df_merged['is_rf_residence'])]
rf_total = len(rf)
rf_children = rf['is_child'].sum()
rf_essay = len(rf[rf['Статус эссе'].notna()])
rf_video = len(rf[rf['Статус видеовизитки'].notna()])
rf_children_essay = len(rf[(rf['Статус эссе'].notna()) & (rf['is_child'])])
rf_children_video = len(rf[(rf['Статус видеовизитки'].notna()) & (rf['is_child'])])

In [46]:
# ИН
foreign = df_merged[~df_merged['is_rf_citizenship'] & ~df_merged['is_rf_residence']]
foreign_total = len(foreign)
foreign_children = foreign['is_child'].sum()
foreign_essay = len(foreign[foreign['Статус эссе'].notna()])
foreign_video = len(foreign[foreign['Статус видеовизитки'].notna()])
foreign_children_essay = len(foreign[(foreign['Статус эссе'].notna()) & (foreign['is_child'])])
foreign_children_video = len(foreign[(foreign['Статус видеовизитки'].notna()) & (foreign['is_child'])])

In [47]:
# ИН в РФ
foreign_in_rf = df_merged[~df_merged['is_rf_citizenship'] & df_merged['is_rf_residence']]
foreign_in_rf_total = len(foreign_in_rf)
foreign_in_rf_children = foreign_in_rf['is_child'].sum()
foreign_in_rf_essay = len(foreign_in_rf[foreign_in_rf['Статус эссе'].notna()])
foreign_in_rf_video = len(foreign_in_rf[foreign_in_rf['Статус видеовизитки'].notna()])
foreign_in_rf_children_essay = len(foreign_in_rf[(foreign_in_rf['Статус эссе'].notna()) & (foreign_in_rf['is_child'])])
foreign_in_rf_children_video = len(foreign_in_rf[(foreign_in_rf['Статус видеовизитки'].notna()) & (foreign_in_rf['is_child'])])

In [48]:
# Соотечественники
compatriots = df_merged[df_merged['is_rf_citizenship'] & ~df_merged['is_rf_residence']]
compatriots_total = len(compatriots)
compatriots_children = compatriots['is_child'].sum()
compatriots_essay = len(compatriots[compatriots['Статус эссе'].notna()])
compatriots_video = len(compatriots[compatriots['Статус видеовизитки'].notna()])
compatriots_children_essay = len(compatriots[(compatriots['Статус эссе'].notna()) & (compatriots['is_child'])])
compatriots_children_video = len(compatriots[(compatriots['Статус видеовизитки'].notna()) & (compatriots['is_child'])])

In [49]:
# Количество стран
countries_count = df_merged['Страна проживания'].nunique()

In [50]:
print(f"""
**ЗАЯВОЧНАЯ КАМПАНИЯ | {datetime.now().strftime('%d.%m.%Y, %H:%M')}**

**Всего** — {total} заявок, из них:
14-17 лет — {children_14_17} чел.,
18-35 лет — {adults_18_35} чел.

**Отправлено эссе** — {total_essay}
14-17 лет — {essay_children} эссе,
18-35 лет — {essay_adults} эссе

**Отправлено видео** — {total_video}
14-17 лет — {video_children} видео,
18-35 лет — {video_adults} видео

🇷🇺 **РФ** — {rf_total} чел.,
**Отправлено эссе** — {rf_essay},
**Отправлено видео** — {rf_video}
🔵 **из них дети (14-17)** — {rf_children} чел.
**Отправлено эссе** — {rf_children_essay},
**Отправлено видео** — {rf_children_video}

🇷🇺 **ИН** — {foreign_total} чел.,
**Отправлено эссе** — {foreign_essay},
**Отправлено видео** — {foreign_video}
🔵 **из них дети (14-17)** — {foreign_children} чел.
**Отправлено эссе** — {foreign_children_essay},
**Отправлено видео** — {foreign_children_video}

🇷🇺 **ИН в РФ** — {foreign_in_rf_total} чел.,
**Отправлено эссе** — {foreign_in_rf_essay},
**Отправлено видео** — {foreign_in_rf_video}
🔵 **из них дети (14-17)** — {foreign_in_rf_children} чел.
**Отправлено эссе** — {foreign_in_rf_children_essay},
**Отправлено видео** — {foreign_in_rf_children_video}

🇷🇺 **Соотечественники** — {compatriots_total} чел.,
**Отправлено эссе** — {compatriots_essay},
**Отправлено видео** — {compatriots_video}
🔵 **из них дети (14-17)** — {compatriots_children} чел.
**Отправлено эссе** — {compatriots_children_essay},
**Отправлено видео** — {compatriots_children_video}

🌍 **Количество стран** — {countries_count}
""")


**ЗАЯВОЧНАЯ КАМПАНИЯ | 16.03.2026, 16:41**

**Всего** — 33731 заявок, из них:
14-17 лет — 5114 чел.,
18-35 лет — 28617 чел.

**Отправлено эссе** — 5186
14-17 лет — 720 эссе,
18-35 лет — 4466 эссе

**Отправлено видео** — 996
14-17 лет — 150 видео,
18-35 лет — 846 видео

🇷🇺 **РФ** — 15094 чел.,
**Отправлено эссе** — 2077,
**Отправлено видео** — 445
🔵 **из них дети (14-17)** — 3908 чел.
**Отправлено эссе** — 474,
**Отправлено видео** — 81

🇷🇺 **ИН** — 16957 чел.,
**Отправлено эссе** — 2774,
**Отправлено видео** — 461
🔵 **из них дети (14-17)** — 1152 чел.
**Отправлено эссе** — 232,
**Отправлено видео** — 64

🇷🇺 **ИН в РФ** — 1528 чел.,
**Отправлено эссе** — 307,
**Отправлено видео** — 82
🔵 **из них дети (14-17)** — 11 чел.
**Отправлено эссе** — 1,
**Отправлено видео** — 0

🇷🇺 **Соотечественники** — 152 чел.,
**Отправлено эссе** — 28,
**Отправлено видео** — 8
🔵 **из них дети (14-17)** — 43 чел.
**Отправлено эссе** — 13,
**Отправлено видео** — 5

🌍 **Количество стран** — 169



In [51]:
df_merged['Макрорегион'] = df_merged['Страна проживания'].map(COUNTRY_MACRO_REGIONS).fillna('Нет макро')
df_merged['Федеральный округ'] = df_merged['Регион проживания'].map(RUSSIAN_REGIONS).fillna('Нет ФО')

In [52]:
def category(age):
    if 14 <= age <= 17:
        return '14-17'
    elif 18 <= age <= 35:
        return '18-35'
    else:
        return 'Другая категория'  # или 'Не подходит'

df_merged['Категория участника'] = df_merged['Возраст'].apply(category)

In [53]:
df_merged['Дата создания заявки'] = pd.to_datetime(df_merged['Дата создания заявки'], errors='coerce').dt.strftime("%d.%m.%Y")

In [54]:
def is_profile_complete(row):
    # Список обязательных полей из required_fields.py (3-22)
    required_fields = [
        'ID',
        'Email #email',
        'Гражданство',
        'Страна проживания',
        'Пол #sex',
        'Дата рождения',
        'Дата рождения',
        'Номер телефона #phone',
        'Фотография #photo',
        'Город фактического места проживания #cityresidence',
        'Адрес фактического места проживания #addressresidence',
        'Родной язык #nativelanguage',
        'Язык переписки #correspondencelanguage',
        'Тип питания #dietype',
        'Род деятельность #statusactivity',
        'Направление деятельности #fieldwork',
        'Уровень владения английским языком #englevel',
        'Уровень владения русским языком #ruslevel'
    ]
    # Группы "или-или" для ФИО
    either_or_groups = [
        (["Имя на русском #firstName.ru", "Имя на английском #firstName.en", "Нет имени"], "Нет имени"),
        (["Фамилия на русском #lastName.ru", "Фамилия на английском #lastName.en", "Нет фамилии"], "Нет фамилии"),
        (["Отчество на русском #patronymic.ru", "Отчество на английском #patronymic.en", "Нет отчества"], "Нет отчества"),
    ]
    # Проверяем обязательные поля
    for field in required_fields:
        if pd.isna(row.get(field)) or str(row.get(field)).strip() == '':
            return False
    # Проверяем либо хотя бы одно из имен (на русском/английском), либо "Нет имени" == "Да"
    for fields, neg_field in either_or_groups:
        # индексы 0,1 - это имя/фамилия/отчество, 2 - "Нет имя" и т.д.
        if (
            (pd.isna(row.get(fields[0])) or str(row.get(fields[0])).strip() == '') and
            (pd.isna(row.get(fields[1])) or str(row.get(fields[1])).strip() == '')
        ):
            # Если оба пустые, то проверяем "Нет ..."
            if str(row.get(fields[2])).strip() != 'Да':
                return False
    return True

df_merged['Профиль заполнен полностью'] = df_merged.apply(is_profile_complete, axis=1)

In [55]:
# import random

# required_fields = [
#         'ID',
#         'Email #email',
#         'Гражданство',
#         'Страна проживания',
#         'Регион проживания',
#         'Пол #sex',
#         'Дата рождения',
#         'Дата рождения',
#         'Номер телефона #phone',
#         'Фотография #photo',
#         'Город фактического места проживания #cityresidence',
#         'Адрес фактического места проживания #addressresidence',
#         'Родной язык #nativelanguage',
#         'Язык переписки #correspondencelanguage',
#         'Тип питания #dietype',
#         'Род деятельность #statusactivity',
#         'Учусь в России #studyrussia',
#         'Направление деятельности #fieldwork',
#         'Уровень владения английским языком #englevel',
#         'Уровень владения русским языком #ruslevel'
#     ]
#     # Группы "или-или" для ФИО
# either_or_groups = [
#         (["Имя на русском #firstName.ru", "Имя на английском #firstName.en", "Нет имени"], "Нет имени"),
#         (["Фамилия на русском #lastName.ru", "Фамилия на английском #lastName.en", "Нет фамилии"], "Нет фамилии"),
#         (["Отчество на русском #patronymic.ru", "Отчество на английском #patronymic.en", "Нет отчества"], "Нет отчества"),
#     ]

# # Выберем 5 случайных индексов
# random_indices = random.sample(list(df_merged.index), 5)

# for idx in random_indices:
#     row = df_merged.loc[idx]
#     missing_fields = []
#     # Проверяем обязательные поля
#     for field in required_fields:
#         if pd.isna(row.get(field)) or str(row.get(field)).strip() == '':
#             missing_fields.append(field)
#     # Проверяем "или-или" группы для ФИО
#     for fields, neg_field in either_or_groups:
#         if (
#             (pd.isna(row.get(fields[0])) or str(row.get(fields[0])).strip() == '') and
#             (pd.isna(row.get(fields[1])) or str(row.get(fields[1])).strip() == '')
#         ):
#             if str(row.get(fields[2])).strip() != 'Да':
#                 # Добавляем недостающие поля: оба имени и нет "Да" в "Нет имя"
#                 missing_fields.append(f"{fields[0]} или {fields[1]} или '{fields[2]}' == 'Да'")
#     print(f"\nСтрока c индексом {idx}:")
#     print(row)
#     if missing_fields:
#         print("Недостающие поля для полного профиля:")
#         for fld in missing_fields:
#             print(f"  - {fld}")
#     else:
#         print("Все необходимые поля заполнены, профиль считается полным.")

In [56]:
# pd.set_option('display.max_columns', None)
# display(df_merged)

In [57]:
# # Получим список пользователей с неполным профилем и статусом "На рассмотрении"
# incomplete_review = df_merged[(df_merged['Профиль заполнен полностью'] == False) & (df_merged['Статус заявки'] == "Черновик")]

# # Список обязательных полей
# from required_fields import rules
# required_fields = rules['required']

# # Поля "или-или" для ФИО
# either_or_groups = rules['either_or']

# def get_missing_fields(row):
#     missing = []
#     # Обычные обязательные поля
#     for field in required_fields:
#         if pd.isna(row.get(field)) or str(row.get(field)).strip() == '':
#             missing.append(field)
#     # Проверка "или-или"-групп
#     for fields in either_or_groups:
#         # например: ['Имя на русском', 'Имя на английском', 'Нет имени']
#         field1, field2, none_field = fields
#         f1_empty = pd.isna(row.get(field1)) or str(row.get(field1)).strip() == ''
#         f2_empty = pd.isna(row.get(field2)) or str(row.get(field2)).strip() == ''
#         none_yes = (str(row.get(none_field)).strip() == 'Да')
#         if f1_empty and f2_empty and not none_yes:
#             # Нужно заполнить хотя бы один, если нет "Нет <field>" == "Да"
#             missing.append(f"{field1} или {field2} или '{none_field}'=='Да'")
#     return missing

# # Для каждой строки покажем недостающие поля
# for idx, row in incomplete_review.iterrows():
#     missing = get_missing_fields(row)
#     print(f"Строка {idx} (ID: {row.get('ID')}):")
#     if missing:
#         print("  Не заполнены поля:")




#         for m in missing:
#             print(f"   - {m}")
#     else:
#         print("  Причина неясна — все обязательные поля заполнены.")

In [58]:
df_merged[(df_merged['Страна проживания'] == "Венесуэла") & (df_merged['Возраст'].between(14,17))]

,ID,Email #email,Гражданство,Страна проживания,Регион проживания,Возраст,Готов добраться за свой счёт #travelownexpense,Участие в РП (да/нет) #expeditionprogram,Дата создания заявки,Имя на русском #firstName.ru,...,is_child,is_rf_citizenship,is_rf_residence,Статус эссе,Статус видеовизитки,Статус заявки,Макрорегион,Федеральный округ,Категория участника,Профиль заполнен полностью
1480,d39499c2-c5fd-4130-88f4-0a0f41fe1043,joharojas201210@gmail.com,Венесуэла,Венесуэла,NaN,15.0,NaN,NaN,05.02.2026,Джоанна,...,True,False,False,Отправлена,Отправлена,На рассмотрении,Америка,Нет ФО,14-17,True
13356,2a3f291a-21ca-45e1-bfac-a13a4fea1794,nanjeligarciagallardo@gmail.com,Венесуэла,Венесуэла,NaN,16.0,NaN,NaN,14.02.2026,Нанйели,...,True,False,False,Отправлена,NaN,На рассмотрении,Америка,Нет ФО,14-17,True
31956,533fe5fe-54f1-409a-8def-8ac0fb997747,marleyalondravelasquezgomez@gmail.com,Венесуэла,Венесуэла,NaN,17.0,NaN,NaN,13.03.2026,Марли,...,True,False,False,NaN,NaN,Черновик,Америка,Нет ФО,14-17,True


In [59]:
df_merged['Профиль заполнен полностью'].value_counts()

Профиль заполнен полностью
True     25367
False     8364
Name: count, dtype: int64

In [60]:
df_merged['Статус заявки'].value_counts()

Статус заявки
На рассмотрении    22151
Черновик           11580
Name: count, dtype: int64

In [61]:
df_merged[[
    'ID',
    'Гражданство',
    'Страна проживания',
    'Регион проживания',
    'Возраст',
    'Статус заявки',
    'is_child',
    'is_rf_citizenship',
    'is_rf_residence',
    'Готов добраться за свой счёт #travelownexpense',
    'Участие в РП (да/нет) #expeditionprogram',
    'Макрорегион',
    'Федеральный округ',
    'Статус эссе',
    'Статус видеовизитки',
    'Категория участника',
    'Дата создания заявки',
    'Профиль заполнен полностью'
]].to_excel(
    r"outdata/Выгрузка данных от {0}.xlsx".format(date.today()),
    index=False
)

In [30]:
df = pd.read_excel('outdata/Выгрузка данных от 2026-03-16.xlsx')

In [31]:
df.

,ID,Гражданство,Страна проживания,Регион проживания,Возраст,Статус заявки,is_child,is_rf_citizenship,is_rf_residence,Готов добраться за свой счёт #travelownexpense,Участие в РП (да/нет) #expeditionprogram,Макрорегион,Федеральный округ,Статус эссе,Статус видеовизитки,Категория участника,Дата создания заявки,Профиль заполнен полностью
0,30aea6f5-91b0-425f-be3f-9af5f76083cf,Вьетнам,Германия,NaN,26,Черновик,False,False,False,Да,Да,Европа,Нет ФО,NaN,NaN,18-35,04.02.2026,True
1,d864a015-2e42-406e-b274-6b16f17c43e7,Таджикистан,Россия,Московская область,28,На рассмотрении,False,False,True,NaN,NaN,Нет макро,ЦФО,Отправлена,NaN,18-35,05.02.2026,True
2,1d2be7e1-9f4d-48b1-923e-44a627caa682,Афганистан,Афганистан,NaN,30,На рассмотрении,False,False,False,Да,Да,Азия и Океания,Нет ФО,Отправлена,NaN,18-35,05.02.2026,True
3,7df73f1e-2791-41c4-b462-0f9806ceff7d,Россия,Россия,Свердловская область,19,Черновик,False,True,True,NaN,NaN,Нет макро,УФО,NaN,NaN,18-35,05.02.2026,True
4,25247c8c-72db-4207-ae40-8af5267d79ed,Алжир,Россия,Волгоградская область,21,На рассмотрении,False,False,True,NaN,NaN,Нет макро,ЮФО,Отправлена,NaN,18-35,05.02.2026,True


In [33]:
df[df['Страна проживания'] != 'Россия'].count()

ID                                                17027
Гражданство                                       17025
Страна проживания                                 17026
Регион проживания                                     0
Возраст                                           17027
Статус заявки                                     17027
is_child                                          17027
is_rf_citizenship                                 17027
is_rf_residence                                   17027
Готов добраться за свой счёт #travelownexpense    11596
Участие в РП (да/нет) #expeditionprogram          11548
Макрорегион                                       17027
Федеральный округ                                 17027
Статус эссе                                        2808
Статус видеовизитки                                 489
Категория участника                               17027
Дата создания заявки                              17027
Профиль заполнен полностью                      